<a href="https://colab.research.google.com/github/ian-ren03/NLP-TC3009-A01384161/blob/main/notebooks/02_modelo_b_transformer_compacto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modelo B: Transformer Compacto (Desde Cero) — NMT EN → ES

**Proyecto Final Integrador — Procesamiento de Lenguaje Natural (TC3009C)**

- **Arquitectura:** Encoder–Decoder Transformer (`batch_first=True`) con *Multi-Head Self-Attention*, *Cross-Attention* y `PositionalEmbedding` corregido de forma `(1, max_seq_len, d_model)`.
- **Hiperparámetros (Sección 4):** `d_model = 128`, `num_heads = 4`, `d_ff = 512`, `num_layers = 2`, `dropout = 0.1`.
- **Integridad experimental:** Protocolo estricto 80/10/10 (`SHA-256 = dbac04ac5648`), vocabularios construidos únicamente con `train_pairs`, aislamiento absoluto de `test_pairs` y normalización simétrica (`clean_text`) para `sacrebleu`.

## 1. Configuración del entorno (Colab / Local)

In [ ]:
import os
import sys
import random
import numpy as np
import torch

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/PROYECTO FINAL NLP - ntm-en-es'
    if not os.path.exists('NLP-TC3009-A01384161'):
        !git clone https://github.com/ian-ren03/NLP-TC3009-A01384161.git
    %cd NLP-TC3009-A01384161
    !git pull
    !pip install -q sacrebleu
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
    SAVE_DIR = 'checkpoints'

if '.' not in sys.path:
    sys.path.append('.')

os.makedirs(SAVE_DIR, exist_ok=True)

SEED = 23
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Entorno: {"Colab" if IN_COLAB else "Local"} | Dispositivo: {device}')
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Protocolo obligatorio de descarga, filtrado y partición (Listing 1)

In [ ]:
import hashlib
import urllib.request
import zipfile

# 1. Descarga directa (funciona tanto en Colab/Linux como en Windows local)
if not os.path.exists('spa.txt'):
    url = 'https://www.manythings.org/anki/spa-eng.zip'
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as resp, open('spa-eng.zip', 'wb') as out:
        out.write(resp.read())
    with zipfile.ZipFile('spa-eng.zip', 'r') as zf:
        zf.extractall('.')

with open('spa.txt', 'r', encoding='utf-8') as f:
    raw_pairs = [line.strip().split('\t')[:2] for line in f if '\t' in line]

# 2. Filtrado de complejidad mínima (>= 8 palabras en inglés, <= 120 tokens)
filtered_pairs = [
    (en, es) for en, es in raw_pairs
    if 8 <= len(en.split()) <= 120 and len(es.split()) <= 120
]

# 3. Mezcla determinista con semilla fija (obligatoria para auditoría)
rng = random.Random(23)
rng.shuffle(filtered_pairs)

# 4. Partición estricta 80% Train / 10% Validation / 10% Test
n = len(filtered_pairs)
n_train = int(0.80 * n)
n_val = int(0.10 * n)

train_pairs = filtered_pairs[:n_train]
val_pairs = filtered_pairs[n_train:n_train + n_val]
test_pairs = filtered_pairs[n_train + n_val:]

# 5. Huella de auditoría
test_hash = hashlib.sha256("\n".join([p[0] for p in test_pairs]).encode()).hexdigest()[:12]
print(f'Train: {len(train_pairs)} | Val: {len(val_pairs)} | Test: {len(test_pairs)}')
print(f'SHA-256 Test Audit ID: {test_hash}')

EXPECTED_HASH = 'dbac04ac5648'
assert test_hash == EXPECTED_HASH, f'Hash inesperado: {test_hash} != {EXPECTED_HASH}'

## 3. Vocabularios (solo sobre `train_pairs`) y DataLoaders

In [ ]:
from torch.utils.data import DataLoader
from src.data import (
    PAD_IDX, UNK_IDX, SOS_IDX, EOS_IDX,
    clean_text, build_vocab, NMTDataset, collate_fn, BucketSampler
)

# Regla 1: Cero fuga de datos — vocabularios construidos ÚNICAMENTE con train_pairs
eng_word2idx = build_vocab([en for en, _ in train_pairs], min_freq=2, max_size=20000)
spa_word2idx = build_vocab([es for _, es in train_pairs], min_freq=2, max_size=20000)
spa_idx2word = {v: k for k, v in spa_word2idx.items()}

print(f'Vocab EN: {len(eng_word2idx)} | Vocab ES: {len(spa_word2idx)}')

BATCH_SIZE = 64
train_ds = NMTDataset(train_pairs, eng_word2idx, spa_word2idx)
val_ds = NMTDataset(val_pairs, eng_word2idx, spa_word2idx)
test_ds = NMTDataset(test_pairs, eng_word2idx, spa_word2idx)

train_loader = DataLoader(
    train_ds,
    batch_sampler=BucketSampler([len(s) for s in train_ds.src], batch_size=BATCH_SIZE, shuffle=True),
    collate_fn=collate_fn,
)
val_loader = DataLoader(
    val_ds,
    batch_sampler=BucketSampler([len(s) for s in val_ds.src], batch_size=BATCH_SIZE, shuffle=False),
    collate_fn=collate_fn,
)
test_loader = DataLoader(
    test_ds,
    batch_sampler=BucketSampler([len(s) for s in test_ds.src], batch_size=BATCH_SIZE, shuffle=False),
    collate_fn=collate_fn,
)
print(f'Batches -> Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

## 4. Instanciación del Modelo B y verificación de `PositionalEmbedding`

In [ ]:
from src.transformer import TransformerNMT, MODEL_B_CONFIG, count_trainable_parameters

cfg = MODEL_B_CONFIG
model_b = TransformerNMT(
    src_vocab_size=len(eng_word2idx),
    tgt_vocab_size=len(spa_word2idx),
    d_model=cfg['d_model'],
    num_heads=cfg['num_heads'],
    d_ff=cfg['d_ff'],
    num_layers=cfg['num_layers'],
    dropout=cfg['dropout'],
    max_seq_len=cfg['max_seq_len'],
    pad_idx=PAD_IDX,
).to(device)

# Verificación obligatoria de dimensiones en PositionalEmbedding
assert model_b.pos_encoding.pe.shape == (1, cfg['max_seq_len'], cfg['d_model'])
print('Configuración Modelo B:', cfg)
print('Forma de PositionalEmbedding.pe:', tuple(model_b.pos_encoding.pe.shape))
print(f'Parámetros entrenables: {count_trainable_parameters(model_b):,}')

## 5. Entrenamiento (Train vs. Validation Loss)

Entrenamos durante 25 épocas sin truncar prematuramente para observar tanto el punto de convergencia como el inicio del sobreajuste (*overfitting*). El mejor checkpoint se selecciona estrictamente por `val_loss`.

In [ ]:
from src.eval_utils import train_transformer, plot_learning_curves

EPOCHS = 25
LR = 1e-3
optimizer = torch.optim.Adam(model_b.parameters(), lr=LR, betas=(0.9, 0.98), eps=1e-9)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

history_b = train_transformer(
    model=model_b,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    epochs=EPOCHS,
    model_name=cfg['name'],
    save_dir=SAVE_DIR,
    device=device,
    grad_clip=1.0,
    patience=None,
    use_amp=True,
    scheduler=scheduler,
)

plot_learning_curves(
    history_b,
    title='Modelo B (Transformer Compacto) — Train vs. Validation Loss',
    save_path='figures/modelo_b_curvas.png',
)

## 6. Evaluación final exclusiva en `test_pairs` (Cuadro 1)

In [ ]:
import pandas as pd
from src.persist import load_experiment
from src.eval_utils import evaluate_on_test

# Cargar el mejor modelo según Validation Loss
model_b, _ = load_experiment(model_b, f"{cfg['name']}_best", SAVE_DIR, device=device)

summary_b, preds_b = evaluate_on_test(
    model=model_b,
    model_name=cfg['name'],
    test_loader=test_loader,
    test_pairs=test_pairs,
    eng_word2idx=eng_word2idx,
    spa_word2idx=spa_word2idx,
    history=history_b,
    device=device,
    train_pairs=train_pairs,
    results_dir='results',
    batch_size=BATCH_SIZE,
    max_len=64,
)

tabla_b = pd.DataFrame([{
    'Corpus BLEU Global': summary_b['bleu_global'],
    'BLEU Medias (8-18 pal.)': summary_b['bleu_medium_8_18'],
    'BLEU Largas (>18 pal.)': summary_b['bleu_long_gt_18'],
    'BLEU Test No Visto': summary_b['bleu_unseen_test'],
    'Test Loss': summary_b['test_loss'],
    'Test Perplejidad': summary_b['test_perplexity'],
    'Parámetros Entrenables': f"{summary_b['trainable_params']:,}",
    'Tiempo / Época (s)': summary_b['avg_epoch_time_s'],
    'Latencia Inferencia (ms/oración)': summary_b['inference_latency_ms'],
}], index=['Modelo B (Transf. Comp.)']).T
display(tabla_b)

## 7. Visualización de Pesos de Atención (Heatmap) y Análisis Cualitativo

In [ ]:
from src.eval_utils import plot_attention_heatmap

print('=== 5 Ejemplos Cualitativos del Conjunto Test ===\n')
for k, ex in enumerate(summary_b['qualitative_examples'], 1):
    print(f"[{k}] (idx={ex['test_idx']}, {ex['src_words']} palabras EN)")
    print(f"  EN : {ex['src_en']}")
    print(f"  REF: {ex['ref_es']}")
    print(f"  MOD B: {ex['pred_es']}\n")

# Heatmap de Cross-Attention sobre una oración del conjunto de prueba
ej_heatmap = summary_b['qualitative_examples'][0]
pred_hm = plot_attention_heatmap(
    model=model_b,
    sentence_en=ej_heatmap['src_en'],
    eng_word2idx=eng_word2idx,
    spa_idx2word=spa_idx2word,
    device=device,
    title='Modelo B (Transformer Compacto) — Pesos de Cross-Attention',
    save_path='figures/modelo_b_atencion.png',
)
print('Traducción visualizada:', pred_hm)